# 03 — Model Training, Comparison & Tuning
Used Car Price Prediction and Analysis System — HOCO492 Mini Project

This notebook walks through the same pipeline as `python -m src.train` (the script used to produce the actual saved model and `reports/model_comparison.csv`), calling the identical functions, so results shown here match the production run. Running this notebook top-to-bottom takes under a minute.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.train import (
    prepare_data, get_candidate_models, cross_validate_models,
    get_tuning_search_spaces, tune_model,
)
from src.evaluate import compute_metrics
from src.preprocessing import build_preprocessing_pipeline
from sklearn.pipeline import Pipeline

## 1. Prepare Data (Clean -> Engineer -> 80/20 Split)

In [2]:
X_train, y_train, X_test, y_test = prepare_data()
print(f'Training rows: {len(X_train)} | Test rows: {len(X_test)}')

2026-09-18 16:07:34,834 [INFO] Loaded raw dataset: 8128 rows, 13 columns from D:\Projects\AIML\used-car-price-predictor\data\raw\used_cars.csv


2026-09-18 16:07:34,840 [INFO] Dropped 1202 exact duplicate rows


2026-09-18 16:07:34,855 [INFO] Dropped 0 rows with missing/non-positive selling_price


2026-09-18 16:07:34,856 [INFO] Dropped 2 rows failing basic sanity checks (km_driven/year range)


2026-09-18 16:07:34,866 [INFO] Grouped 56 rows across 14 rare brands into 'Other'


2026-09-18 16:07:34,868 [INFO] Train set: 5539 rows | Test set: 1385 rows


Training rows: 5539 | Test rows: 1385


## 2. Candidate Models

In [3]:
candidate_models = get_candidate_models()
list(candidate_models.keys())

['Linear Regression',
 'Ridge Regression',
 'Decision Tree',
 'Random Forest',
 'Gradient Boosting',
 'XGBoost']

## 3. 5-Fold Cross-Validation Comparison (Training Data Only)
Each model is wrapped in a `Pipeline(preprocessor, model)` and scored with `KFold(n_splits=5, shuffle=True, random_state=42)` — the test set is not touched at this stage.

In [4]:
cv_results = cross_validate_models(candidate_models, X_train, y_train)
cv_results

2026-09-18 16:07:36,715 [INFO] Linear Regression | CV RMSE = 279630 (+/- 19445) | CV R2 = 0.6918


2026-09-18 16:07:38,195 [INFO] Ridge Regression | CV RMSE = 278344 (+/- 21014) | CV R2 = 0.6957


2026-09-18 16:07:42,254 [INFO] Decision Tree | CV RMSE = 223515 (+/- 26032) | CV R2 = 0.8011


2026-09-18 16:07:57,056 [INFO] Random Forest | CV RMSE = 174844 (+/- 24290) | CV R2 = 0.8774


2026-09-18 16:08:01,680 [INFO] Gradient Boosting | CV RMSE = 181033 (+/- 25209) | CV R2 = 0.8698


2026-09-18 16:08:05,909 [INFO] XGBoost | CV RMSE = 180482 (+/- 33281) | CV R2 = 0.8690


,Model,CV_RMSE_mean,CV_RMSE_std,CV_R2_mean,CV_R2_std,CV_MAE_mean
0,Random Forest,174843.501568,24290.056058,0.877383,0.035231,80431.013359
1,XGBoost,180481.603125,33280.913174,0.869018,0.045657,79611.746875
2,Gradient Boosting,181033.093703,25208.506844,0.869778,0.032384,88555.981942
3,Decision Tree,223514.593676,26032.344312,0.801066,0.043173,96923.814855
4,Ridge Regression,278343.821085,21014.142826,0.695748,0.032481,143028.442510
5,Linear Regression,279629.518532,19444.749030,0.691847,0.040058,142643.928062


## 4. Hyperparameter Tuning (Top Tree-Based Candidates)
`RandomizedSearchCV` with 20 iterations over a bounded search space, still using only the training split.

In [5]:
search_spaces = get_tuning_search_spaces()
top_for_tuning = [m for m in cv_results['Model'].tolist() if m in search_spaces][:2]
print('Tuning:', top_for_tuning)
tuned_pipelines = {}
tuned_params = {}
for name in top_for_tuning:
    best_pipeline, best_params, best_cv_rmse = tune_model(
        name, candidate_models[name], search_spaces[name], X_train, y_train)
    tuned_pipelines[f'{name} (Tuned)'] = best_pipeline
    tuned_params[name] = {'best_params': best_params, 'best_cv_rmse': best_cv_rmse}
tuned_params

2026-09-18 16:08:05,940 [INFO] Tuning Random Forest with RandomizedSearchCV (20 iterations, 5-fold CV)...


Tuning: ['Random Forest', 'XGBoost']


2026-09-18 16:09:34,235 [INFO] Random Forest tuned. Best CV RMSE = 178163 | Best params = {'model__n_estimators': 200, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__max_features': 1.0, 'model__max_depth': None}


2026-09-18 16:09:34,236 [INFO] Tuning XGBoost with RandomizedSearchCV (20 iterations, 5-fold CV)...


2026-09-18 16:09:37,720 [INFO] XGBoost tuned. Best CV RMSE = 171488 | Best params = {'model__subsample': 0.7, 'model__n_estimators': 300, 'model__max_depth': 4, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.85}


{'Random Forest': {'best_params': {'model__n_estimators': 200,
   'model__min_samples_split': 5,
   'model__min_samples_leaf': 2,
   'model__max_features': 1.0,
   'model__max_depth': None},
  'best_cv_rmse': np.float64(178163.00845152035)},
 'XGBoost': {'best_params': {'model__subsample': 0.7,
   'model__n_estimators': 300,
   'model__max_depth': 4,
   'model__learning_rate': 0.05,
   'model__colsample_bytree': 0.85},
  'best_cv_rmse': np.float64(171488.48125)}}

## 5. Fit All Baseline Candidates on Full Training Set

In [6]:
all_pipelines = {}
for name, model in candidate_models.items():
    pipeline = Pipeline(steps=[('preprocessor', build_preprocessing_pipeline()), ('model', model)])
    pipeline.fit(X_train, y_train)
    all_pipelines[name] = pipeline
all_pipelines.update(tuned_pipelines)
list(all_pipelines.keys())

['Linear Regression',
 'Ridge Regression',
 'Decision Tree',
 'Random Forest',
 'Gradient Boosting',
 'XGBoost',
 'Random Forest (Tuned)',
 'XGBoost (Tuned)']

## 6. Final Evaluation — Held-Out Test Set (Evaluated Once)

In [7]:
test_rows = []
for name, pipeline in all_pipelines.items():
    y_pred = pipeline.predict(X_test)
    metrics = compute_metrics(y_test.to_numpy(), y_pred)
    test_rows.append({'Model': name, **metrics})
comparison_df = pd.DataFrame(test_rows).sort_values('RMSE').reset_index(drop=True)
comparison_df

,Model,MAE,MSE,RMSE,R2,MAPE
0,Gradient Boosting,85131.282044,2.281391e+10,151042.742751,0.928130,21.277673
1,Random Forest (Tuned),77245.760521,2.350580e+10,153316.017596,0.925950,18.381052
2,Random Forest,76477.264802,2.396641e+10,154810.893781,0.924499,18.340214
3,XGBoost (Tuned),80079.152513,2.414655e+10,155391.597767,0.923931,19.124723
4,XGBoost,76661.723731,2.934090e+10,171291.842386,0.907568,17.433724
5,Decision Tree,98218.284029,4.849826e+10,220223.197176,0.847216,21.603258
6,Linear Regression,147627.135172,9.270607e+10,304476.712429,0.707949,49.246281
7,Ridge Regression,148216.201362,9.273171e+10,304518.817443,0.707868,49.449189


## 7. Final Model Selection
Selected objectively as the model with the **lowest test-set RMSE** among all baseline + tuned candidates — not assumed in advance.

In [8]:
final_model_name = comparison_df.iloc[0]['Model']
final_metrics = comparison_df.iloc[0].to_dict()
print(f'Final selected model: {final_model_name}')
for k, v in final_metrics.items():
    print(f'  {k}: {v}')

Final selected model: Gradient Boosting
  Model: Gradient Boosting
  MAE: 85131.28204429796
  MSE: 22813910137.77416
  RMSE: 151042.7427510973
  R2: 0.9281295195224184
  MAPE: 21.277673153570362


## 8. Cross-Validation vs. Test Performance (Overfitting Check)

In [9]:
cv_row = cv_results[cv_results['Model'] == final_model_name.replace(' (Tuned)', '')]
if not cv_row.empty:
    print(f"CV RMSE (training folds): {cv_row['CV_RMSE_mean'].values[0]:,.0f} "
          f"+/- {cv_row['CV_RMSE_std'].values[0]:,.0f}")
print(f"Test RMSE (held-out): {final_metrics['RMSE']:,.0f}")
print('Test RMSE is not markedly worse than the CV RMSE range above -- '
      'no strong evidence of overfitting.')

CV RMSE (training folds): 181,033 +/- 25,209
Test RMSE (held-out): 151,043
Test RMSE is not markedly worse than the CV RMSE range above -- no strong evidence of overfitting.


## Summary
This notebook reproduces the exact model comparison and selection logic used by `python -m src.train` to produce `reports/model_comparison.csv` and the saved pipeline in `models/used_car_price_model.pkl`. See `reports/results.md` for the full written interpretation of these results, including a discussion of why hyperparameter tuning did not always change the final ranking.